# Summarization API endpoint

In [ ]:
import json
import os

import requests
from rich.pretty import pprint

In [ ]:
API_BASE_URL = os.getenv("AYMURAI_API_BASE_URL", "http://localhost:8999")
SUMMARIZE_URL = f"{API_BASE_URL}/llm/summarize"
print(f"POST {SUMMARIZE_URL}")

## Build request payload

In [ ]:
sample_text = """
La inteligencia artificial (IA) ha revolucionado numerosos campos, desde la medicina hasta la industria del entretenimiento,
y el ámbito de la justicia no es una excepción. La IA tiene el potencial de transformar la forma en que se administran los
sistemas legales, mejorando la eficiencia y la precisión en la toma de decisiones judiciales. Sin embargo, también plantea
desafíos éticos y legales significativos que deben ser abordados cuidadosamente.
""".strip()

payload = {
    "text": sample_text,
    "model": "llama3.2:3b",
    "system_prompt": "Eres un asistente que resume textos de manera concisa y clara sin inventar información.",
    "tokenizer": None,
    "options": {"num_ctx": 1024, "num_predict": 512, "stream": True},
}

pprint(payload)


## Call the endpoint

In [ ]:
def call_summarize(text_payload: dict) -> dict:
    response = requests.post(SUMMARIZE_URL, json=text_payload)
    response.raise_for_status()
    return response.json()


result = call_summarize(payload)
pprint({"model": result.get("model"), "chunks_used": result.get("chunks_used")})
print("Summary:")
print(result.get("summary", "<no summary returned>"))

print("Steps:")
pprint(result.get("steps", []))

## Streamed call

In [ ]:
STREAM_URL = f"{API_BASE_URL}/llm/summarize/stream"


def stream_summarize(text_payload: dict) -> None:
    with requests.post(STREAM_URL, json=text_payload, stream=True, timeout=120) as resp:
        resp.raise_for_status()
        for raw_line in resp.iter_lines():
            if not raw_line:
                continue
            if not raw_line.startswith(b"data: "):
                continue
            event = json.loads(raw_line.split(b"data: ", 1)[1])
            if event.get("type") == "token":
                print(event.get("text", ""), end="", flush=True)
            elif event.get("type") == "summary":
                print("\n\n---\nFinal summary:\n")
                print(event.get("summary", ""))
                print("\nSteps:", event.get("steps", []))
            else:
                print(f"\n[{event.get('type')}] {event}")


stream_summarize(payload)